# YOLOv11-seg Industrial Defect Segmentation Research Framework
### Google Colab Automated Runner (EXP-00 to EXP-11)

This notebook runs the 12 ablation experiments across 6 representative MVTec AD categories:
- **3 Textures**: `grid`, `carpet`, `tile`
- **3 Objects**: `screw`, `pill`, `bottle`

**Validation Strategy**: Validation Set Approach (80% Train / 20% Val on normal data; fixed standard Test set) with Multi-Seed Variance Analysis (`--seed 42, 1337, 2026`).

## 1. Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/industrial_defect_segmentation'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Persistent Drive workspace ready at: {DRIVE_DIR}")

In [ ]:
# Install core research dependencies
!pip install -q ultralytics pywavelets scipy scikit-learn tabulate huggingface_hub pandas
!nvidia-smi

## 2. Pull / Setup Repository

In [ ]:
# Clone your GitHub repo or copy from Google Drive:
# !git clone <YOUR_GITHUB_REPO_URL> repo && cd repo
import os
if os.path.exists('repo'):
    %cd repo
!pwd
!pytest tests/ -v

## 3. Fast Dataset Fetch (6 Representative Categories from CDN)

In [ ]:
# Uses high-speed CDN to fetch only the 6 required categories directly to ./mvtec_ad/
!python tools/download_mvtec_categories.py

## 4. Primary Research Sweep: All 12 Experiments (Seed 42)

In [ ]:
# 1. Baseline (EXP-00)
!python experiments/run_experiment.py --config configs/experiments/exp00_baseline.yaml

# 2. Group A Synthesizers (EXP-01 CutPaste, EXP-02 DRAEM, EXP-03 NSA)
!python experiments/run_experiment.py --config configs/experiments/exp01_cutpaste.yaml
!python experiments/run_experiment.py --config configs/experiments/exp02_draem.yaml
!python experiments/run_experiment.py --config configs/experiments/exp03_nsa.yaml

# 3. Group B Frequency Preprocessing (EXP-04 FFT, EXP-05 DWT)
!python experiments/run_experiment.py --config configs/experiments/exp04_fft.yaml
!python experiments/run_experiment.py --config configs/experiments/exp05_dwt.yaml

# 4. Group Combinations (EXP-06 CutPaste+FFT, EXP-07 CutPaste+DWT)
!python experiments/run_experiment.py --config configs/experiments/exp06_cutpaste_fft.yaml
!python experiments/run_experiment.py --config configs/experiments/exp07_cutpaste_dwt.yaml

# 5. Group C MVTec-AD-C Optical Robustness Benchmarks (EXP-08 to EXP-11)
!python experiments/run_experiment.py --config configs/experiments/exp08_baseline_corruptions.yaml
!python experiments/run_experiment.py --config configs/experiments/exp09_synthesizer_corruptions.yaml
!python experiments/run_experiment.py --config configs/experiments/exp10_frequency_corruptions.yaml
!python experiments/run_experiment.py --config configs/experiments/exp11_full_system.yaml

## 5. Multi-Seed Variance Analysis (Seeds 1337 & 2026)
Runs top key configurations across multiple seeds to compute standard deviations ($\mu \pm \sigma$).

In [ ]:
REPLICATE_SEEDS = [1337, 2026]
CORE_CONFIGS = [
    "configs/experiments/exp00_baseline.yaml",
    "configs/experiments/exp01_cutpaste.yaml",
    "configs/experiments/exp11_full_system.yaml"
]

for seed in REPLICATE_SEEDS:
    for cfg_path in CORE_CONFIGS:
        print(f"\n>>> Launching Replicate Run: {cfg_path} with Seed {seed} <<<")
        !python experiments/run_experiment.py --config {cfg_path} --seed {seed}

## 6. Consolidate Ablation Matrix & Sync to Google Drive

In [ ]:
# Consolidate all experiment summaries into Markdown and CSV
!python experiments/ablation_matrix.py

# Display summary table
with open('results/ablation_summary.md', 'r') as f:
    print(f.read())

# Copy results, checkpoints, and figures to persistent Google Drive
!cp -r results/ablation_summary.* {DRIVE_DIR}/
!cp -r results/experiments/* {DRIVE_DIR}/
print("All experiment artifacts successfully synchronized to Google Drive!")